# Stable diffusion process
## Imports and setups

In [1]:
import os
from base64 import b64encode

import numpy as np
import torch
from torch import autocast
from torchvision import transforms as trms
from diffusers import AutoencoderKL, LMSDiscreteScheduler, UNet2DConditionModel
from huggingface_hub import notebook_login

# for video display
from IPython.display import HTML
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

from transformers import CLIPTextModel, CLIPTokenizer, logging

from torch import autocast
from torchvision import transforms as tfms
from webob.compat import text_

/opt/homebrew/Caskroom/miniforge/base/envs/transformers/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
torch.manual_seed(0)
if not (Path.home()/'.cache/hugginggface'/'token').exists(): notebook_login()

# supress some unnecesary warnings when loading the CLIPTextModel
logging.set_verbosity_error()

# set device
torch_device = "cuda" if torch.device.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
if "mps" == torch_device: os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

# Loading models

In [ ]:
# load autoencoder model to use to decode the latents into image space
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae")
# load tokenizer and text encoder to tokenize and encode text
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14")
# the UNet model for generating the latents
unet = UNet2DConditionModel.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="unet")
# Noise Scheduler
scheduler = LMSDiscreteScheduler(
    beta_start=0.00085,
    beta_end=0.012,
    beta_schedule="scaled_linear",
    num_train_timesteps=1000
)

# load above to GPU
vae = vae.to(torch_device)
text_encoder = text_encoder.to(torch_device)
unet = unet.to(torch_device)

## Diffusion Loop 

In [ ]:
def set_timesteps(scheduler, num_inference_steps):
    scheduler.set_timesteps(num_inference_steps)
    scheduler.timesteps = scheduler.timesteps.to(torch.float32)

In [ ]:
# define some initial settings
prompt = ["A watercolor paiting of an otter"]
height = 512                        # default height for stable diffusion
width = 512                         # default width for stable diffusion
num_inference_steps = 30            # num of denoising steps
guidance_scale = 7.5                # scale for classifier-free guidance
generator = torch.manual_seed(32)   # seed for initial latent noise
batch_size = 1

# prepare text
text_input = tokenizer(
    prompt,
    padding="max-length",
    max_length=tokenizer.model_max_length,
    truncation=True,
    return_tensors="pt"
)

with torch.no_grad():
    text_embeddings = text_encoder(text_input.input_ids.to(torch_device))

max_length = text_input.input_ids.shape[-1]
uncond_input = tokenizer(
    [""]*batch_size,
    padding="max_length",
    max_length=max_length,
    return_tensors="pt"
)

with torch.no_grad():
    uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))

text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

# prep scheduler
set_timesteps(scheduler, num_inference_steps)

# prepare scheduler
latents = torch.randn(
    (batch_size, unet.in_channels, height // 8, width // 8),
    generator=generator
)
latents = latents.to(torch_device)
latents = latents * scheduler.init_noise_sigma

# now loop
with autocast("cuda"):
    for i, t in tqdm(enumerate(scheduler.timesteps), total=len(scheduler.timesteps)):
        # when doing classifier-free guidance, need to expand the latents
        latent_model_input = torch.cat([latents]*2)
        sigma = scheduler.sigmas[1]
        # scale latents (preconditioning):
        # latent_model_input = latent_model_input / ((sigma**2 + 1) ** 0.5) # Diffusers 0.3 and below
        latent_model_input = scheduler.scale_model_input(latent_model_input, t)

        # predict noise residual
        with torch.no_grad():
            noise_pred = unet(
                latent_model_input,
                t,
                encoder_hidden_states=text_embeddings
            ).sample
        
        # perform guidance
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        # compute the previous noise sample x_t -> x_t-1
        # latents = scheduler.step(noise_pred, i, latents)["prev_sample"] # diffusers 0.3 and below
        latents = scheduler.step(noise_pred, t, latents).prev_sample

# scale and decode the image latents with vae
latents = 1 / 0.18215 * latents
with torch.no_grad():
    image = vae.decode(latents).sample

# display image
image = (image / 2 + 0.5).clamp(0, 1)
image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
images = (image * 255).round().astype("uint8")
pil_images = [Image.fromarray(image) for image in images]
pil_images[0]

## the AutoEncoder part
Creates a latent representation of the image, and then decodes it back by optimizing the reconstruction error.

In [ ]:
def pil_to_latent(input_image):
    # single image -> single latent in a batch (1, 4, 64, 64)
    with torch.no_grad():
        latent = vae.encode(
            tfms.ToTensor()(input_image).unsqueeze(0).to(torch_device)*2-1 # here it scales
        )
        return 0.18215 * latent.latent_dist.sample()

def latents_to_pil(latents):
    # batch of latents -> list of images
    with torch.no_grad():
        image = vae.decode(latents).sample
    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
    imgaes = (image * 255).round().astype("uint8")
    pil_images = [Image.fromarray(image) for image in images]

    return pil_images

Test previous functions and see how it does

In [ ]:
# download an image from the web
!curl --output macaw.jpg 'https://lafeber.com/pet-birds/wp-content/uploads/2018/06/Scarlet-Macaw-2.jpg'

In [ ]:
# load image with PIL
input_image = Image.open("macaw.jpg").resize((512, 512))
input_image

In [ ]:
# encode into latent space
encoded = pil_to_latent(input_image)
encoded.shape

In [ ]:
# visualize the 4 channels of this latent representation
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for c in range(4):
    axs[c].imshow(encoded[0][c].cpu(), cmap="Greys")

Here it can be seen that the latent representation really captures a lot of information, so it can handle the reconstruction almost as equal as original image.

In [ ]:
# create the image back from latents
decoded = latents_to_pil(encoded)[0]
decoded

## The scheduler
This is the part where noise is going to be added. The ammount of noise added follows a certain distribution.

In [ ]:
# set number of sampling steps
set_timesteps(scheduler, 15)

# look at how much noise is added at each step
print(scheduler.sigmas)

In [ ]:
# see how it would look like in an image
noise = torch.rand_like(encoded)
sampling_step = 10
encoded_and_noised = scheduler.add_noise(
    encoded,
    noise,
    timesteps=torch.tensor(
        [scheduler.timesteps[sampling_step]]
    )
)
latents_to_pil(encoded_and_noised.float())[0]

# Image2Image.
Here the idea is to noise an original image, then based on some prompt denoise it and create a new one.

In [ ]:
# Settings (same as before except for the new prompt)
prompt = ["A colorful dancer, nat geo photo"]
height = 512                        # default height of Stable Diffusion
width = 512                         # default width of Stable Diffusion
num_inference_steps = 50            # Number of denoising steps
guidance_scale = 8                  # Scale for classifier-free guidance
generator = torch.manual_seed(32)   # Seed generator to create the inital latent noise
batch_size = 1

# need to prepare text same as done before
text_input = tokenizer(
    prompt,
    padding=True,
    max_length=tokenizer.model_max_length,
    truncation=True,
    return_tensors="pt"
)
with torch.no_grad():
    text_embeddings = text_encoder(text_input.input_ids.to(torch_device))[0]
max_lenght = text_input.input_ids.shape[-1]
uncond_input = tokenizer(
    [""]*batch_size,
    padding="max_length",
    max_length=max_lenght,
    return_tensors="pt"
)
with torch.no_grad():
    uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))[0]
text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

# prepare scheduler
set_timesteps(scheduler, num_inference_steps)
# prepare latents
start_step = 10
start_sigma = scheduler.sigmas[start_step]
noise = torch.rand_like(encoded)
latents = scheduler.add_noise(
    encoded,
    noise,
    timesteps=torch.tensor([scheduler.timesteps[start_step]])
)
latents = latents.to(torch_device).float()

# loop
for i, t in tqdm(enumerate(scheduler.timesteps), total=len(scheduler.timesteps)):
    if i >= start_step:
        # if doing clf-free guidance then expand latents
        latent_model_input = torch.cat([latents]*2)
        sigma = scheduler.sigmas[i]
        latent_model_input = scheduler.scale_model_input(latent_model_input, t)
        
        # pred noise residual
        with torch.no_grad():
            noise_pred = unet(
                latent_model_input,
                t,
                encoder_hidden_states=text_embeddings
            )["sample"]
        
        # perform guidance
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text -noise_pred_uncond)
        
        # compute the previous noisy sample x_t -> x_t-1
        latents = scheduler.step(noise_pred, t, latents).prev_sample

latents_to_pil(latents)[0]

## Explore text embeddings and the embedding pipeline

In [ ]:
# define an example prompt
prompt = "A picture of a puppy"

In [ ]:
# 1. explore tokenization as usual
text_input = tokenizer(
    prompt,
    padding="max_length",
    max_length=tokenizer.model_max_length,
    truncation=True,
    return_tensors="pt"
)
# view tokens
text_input['input_ids'][0]

In [ ]:
# see indicual tokens
for t in text_input['input_ids'][0][:8]: # see only first 8 tokens for brevity, the rest will be the padding token
    print(t, tokenizer.decoder.get(int(t)))

In [ ]:
# grab the ouput embeddings
output_embeddings = text_encoder(text_input.input_ids.to(torch_device))[0]
print('Shape', output_embeddings.shape)
output_embeddings

In [ ]:
# to get these embeddings actually the steps are more simple:
# just grab the text encoder which is the CLIP model
text_encoder.text_model.embeddings

## Token embeddings

In [ ]:
# 1. the first part is to get the token embeddings, then will combine with the
# positional embeddings in step 2.

# define the embedding layer
token_embed_layer = text_encoder.text_model.token_embeddings

# if want to embed a single token, you'll need to pass an input id as a tensor to the
# token embed layer
sample_token_input_id = 6829 # for token puppy 
sample_embedding = token_embed_layer(
    torch.tensor(
        sample_token_input_id,
        device=torch_device
    )
)
print(sample_embedding.shape)

In [ ]:
# repeat the above process with the sample prompt to see how looks the prompt embedding
token_embeddings = token_embed_layer(text_input.input_ids.to(torch_device))
print(token_embeddings.shape) # 1 batch, 77 tokens, 768 values per token

## Positional Embeddings

In [ ]:
# 2. second step to combine with the text tokens embeddings
pos_embed_layer = text_encoder.text_model.embeddings.position_embedding
pos_embed_layer # here will positional embeddings for 77 tokens

In [ ]:
# to see the positional embeddings for the sample prompt let's epxlore
position_ids = text_encoder.text_model.embeddings.position_ids[:, :77]
position_embeddings = pos_embed_layer(position_ids)
print(position_embeddings.shape)
position_embeddings

## Combining token embeddings and position embeddings

In [ ]:
# combining embeddings will depend on how the model was trained originally
# for the model we're using the combination of the embeddings it's just an addition of both tensors
input_embeddings = token_embeddings + position_embeddings
print(input_embeddings.shape)
input_embeddings

In [ ]:
# this process was just a demonstration to see that the text encoder model
# already does this both steps for us
text_encoder.text_model.embeddings(text_input.input_ids.to(torch_device))

## Feed the embeddings through the transformer model
Note that here the model used (which is decoder only) uses `causal_mask` insted of `attention_mask`.</br>
So, let's spot some differences of both attention mechanisms:</br>
* `causal_mask`: as said previously, mostly used in decoder only models, but not resticted to them, cause encoder-decoder models use it as well. But the main task of this is to mask tokens after a given token an ensure 'future' is not leaked.
* `attention_mask`: this attention mechanism as well as the previous one, masks padding tokens.

In [ ]:
def get_output_embeds(input_embeddings):
    # CLIP's text model uses causal mask, prepare it here
    bsz, seq_len = input_embeddings.shape[:2]
    causal_attention_mask = text_encoder.text_model.build_causal_attention_mask(
        bsz,
        seq_len,
        dtype=input_embeddings.dtype,
    )

    # need to pass output_hidden_states=True to get the output embeddings
    # and not just the pooled final predictions
    encoder_outputs = text_encoder.text_model.encoder(
        input_embeds=input_embeddings,
        attention_mask=None,
        causal_attention_mask=causal_attention_mask,
        output_attentions=None,
        output_hidden_states=True,
        return_dict=None
    )

    # as the goal in matter is the output hidden state only
    output = encoder_outputs[0]
    # a final layer normalization is applied to the output embeddings
    output = text_encoder.text_model.final_layer_norm(output)

    return output

In [ ]:
out_embeds_test = get_output_embeds(input_embeddings)
print(out_embeds_test.shape)
out_embeds_test

These match the ouput embeddings from somewhere above.</br>

In [ ]:
prompt = "A picture of a puppy"

# tokenize
text_input = tokenizer(
    prompt,
    padding="max_length",
    max_length=tokenizer.model_max_length,
    truncation=True,
    return_tensors="pt"
)
input_ids = text_input.input_ids.to(torch_device)

# get token embeddings from the text_encoder model
token_embeddings = token_embed_layer(input_ids)

# the new embedding. In this case just the input embedding of token 2368...
replacement_token_embedding = text_encoder.get_input_embeddings()(torch.tensor(2368, device=torch_device))
# insert the new embedding in the token embeddings
token_embeddings[0, torch.where(input_ids[0]==6829)] = replacement_token_embedding.to(torch_device)

# combine with positional embeddings
input_embeddings = token_embeddings + position_embeddings
# feed into final output embeds
modified_out_embeddings = get_output_embeds(input_embeddings)

The token replaced was the related to `puppy` which had the `input_id=6829` and was replaced with another `input_id=2368`. An image can be generated using this new embeddings and see what's in result of that replacement.

In [ ]:
# define a new function with these modified embeddings
def generate_with_embeds(text_embeddings):
    height = 512
    width = 512
    num_inference_steps = 30
    guidance_scale = 7.5
    generator = torch.manual_seed(32)
    batch_size = 1

    max_lenght = text_input.input_ids.shape[-1]
    uncond_input = tokenizer(
        [""]*batch_size,
        padding="max_length",
        max_length=max_lenght,
        return_tensors="pt"
    )
    with torch.no_grad():
        uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))[0]
    text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

    # prepare scheduler
    latents = torch.randn(
        (batch_size, unet.in_channels, height // 8, width // 8),
        generator=generator
    )
    latents = latents.to(torch_device)
    latents = latents * scheduler.init_noise_sigma

    # Loop
    for i, t in tqdm(enumerate(scheduler.timesteps), total=len(scheduler.timesteps)):
        latent_model_input = torch.cat([latents]*2)
        sigma = scheduler.sigmas[1]
        latent_model_input = scheduler.scale_model_input(latent_model_input, t)

        # predict residual noise
        with torch.no_grad():
            noise_pred = unet(
                latent_model_input,
                t,
                encoder_hidden_states=text_embeddings
            )["sample"]
        
        # perform guidance
        noise_pred_unconde, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        # compute the previous noise sample x_t -> x_t-1
        latents = scheduler.step(noise_pred, t, latents).prev_sample

    return latents_to_pil(latents)[0]

In [ ]:
generate_with_embeds(modified_out_embeddings)

Another usage based on this aproximation, is combine prompts, as following:

In [ ]:
# let's get a input id from a new prompt (just a single token)
new_prompt = "skunk"
print(tokenizer(new_prompt)) # as a single token was passed, the id will be in the middle of the shown array

In [ ]:
# so combine our intial prompt with the new token
prompt = "A picture of a puppy"
# 1. tokenize
text_input = tokenizer(
    prompt,
    padding="max_length",
    max_length=tokenizer.model_max_length,
    truncation=True,
    return_tensors="pt"
)
input_ids = text_input.input_ids.to(torch_device)

# 2. Get embeddings
token_embeddings = token_embed_layer(input_ids)

# 3. Get the new mixture of embeddings based on prompt and new token
puppy_token_embedding = token_embed_layer(torch.tensor(6829, device=torch_device))
skunk_token_embedding = token_embed_layer(torch.tensor(42194, device=torch_device))
replacement_token_embedding = 0.5*puppy_token_embedding + 0.5*skunk_token_embedding

# 4. Insert new embedding where 'puppy' embedding was
token_embeddings[0, torch.where(input_ids[0]==6829)] = replacement_token_embedding.to(torch_device)

# 5. Combine with positional embeddings
input_embeddings = token_embeddings + position_embeddings

# 6. Get output embeddings
modified_out_embeddings = get_output_embeds(input_embeddings)

# 7. Generate image
generate_with_embeds(modified_out_embeddings)

## Textual Inversion
Use an image to 'learn' a new concept and create a new token embedding.</br>
For this example we will use the `birb-style` learned emebddings to generate images based on a 'learned' concept. 

In [ ]:
birb_embed = torch.load("birb_embeds.bin")
birb_embed.keys(), birb_embed["<birb-style>"].shape

In [ ]:
# now the goal is to replace the token 'pupy' with the birb embeddings
prompt = 'A mouse in the style of puppy'

# 1. tokenize
text_input = tokenizer(
    prompt,
    padding="max_length",
    max_length=tokenizer.model_max_length,
    truncation=True,
    return_tensors="pt"
)
input_ids = text_input.input_ids.to(torch_device)

# 2. Get embeddings
token_embeddings = token_embed_layer(input_ids)
# get the replacement embedding which is going to be the birb style ones
replacement_token_embedding = birb_embed["<birb-style>"].to(torch_device)
# insert into puppy token
token_embeddings[0, torch.where(input_ids[0]==6829)] = replacement_token_embedding.to(torch_device)

# 3. Combine with positional embeddings
input_embeddings = token_embeddings + position_embeddings

# 4. Get output embeddings
modified_out_embeddings = get_output_embeds(input_embeddings)

# 5. Generate image
generate_with_embeds(modified_out_embeddings)

# Further combinations
Now the idea is to generate an image, that'll be a combination of some multiple words prompts.

In [ ]:
# 1. Embed two prompts
text_input1 = tokenizer(
    ["A mouse"],
    padding="max_length",
    max_length=tokenizer.model_max_length,
    truncation=True,
    return_tensors="pt"
)
text_input2 = tokenizer(
    ["A leopard"],
    padding="max_length",
    max_length=tokenizer.model_max_length,
    truncation=True,
    return_tensors="pt"
)
with torch.no_grad():
    text_embeddings1 = text_encoder(text_input1.input_ids.to(torch_device))[0]
    text_embeddings2 = text_encoder(text_input2.input_ids.to(torch_device))[0]

# 2. Combine the embeddings
mix_factor = 0.35 # this will a ponderation of what proportion of each embedding to combine
mixed_embeddings = (
    text_embeddings1 * mix_factor + text_embeddings2 * (1 - mix_factor)
)
# note that the (1-mix_factor) is just to make sure the sum of the factors is 1

# 3. generate image
generate_with_embeds(mixed_embeddings)